In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab_Notebooks/semester_project
%ls

In [ ]:
!pip install chronos-forecasting

In [ ]:
import pandas as pd
from chronos import Chronos2Pipeline
import torch
import numpy as np

In [ ]:
df = pd.read_csv("../../data/EWZ_Daily_Preprocessed.csv")


In [ ]:
df.rename(columns={"Name": "timestamp"}, inplace=True)

df.head()

In [ ]:
split_idx = int(len(df) * 0.85) +1

# Split the dataframe
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

train_df.drop(columns='timestamp', inplace=True)
test_df.drop(columns='timestamp', inplace=True)


In [ ]:
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

In [ ]:

train_values = train_df.values

mean_train = np.mean(train_values, axis=0)
std_train = np.std(train_values, axis=0)

train_values = (train_values - mean_train) / std_train

context = torch.tensor(train_values.T, dtype=torch.float32)
N, T = context.shape

context = context.reshape(N, 1, T)

# Generate multivariate forecast
forecast = np.array(pipeline.predict(
    inputs=context,
    prediction_length=len(test_df),
))

In [ ]:
forecast = np.mean(forecast, axis=2)

In [ ]:
y_true = test_df.values

y_true = (y_true - mean_train) / std_train

y_true = y_true.T

N, T = y_true.shape
y_true = y_true.reshape(N, 1, T)


In [ ]:
np.save("./normalized_pred_chronos.npy", forecast)
np.save("./normalized_true_chronos.npy", y_true)